# iSCORS-Net — Fast Runner

**Workflow:**
1. Run **Setup** — clones the repo and installs deps.
2. Run **Train** — internal learning on the test video.
3. Run **Results** — inline visualisation.
4. Run **Download** — saves `results_<VERSION>.zip` to your machine.

---

In [ ]:
VERSION = 'v3.0'
print(f'iSCORS-Net {VERSION}')

In [ ]:
import os

REPO   = 'https://github.com/breezy90126/iscors-net.git'
BRANCH = 'claude/beautiful-volta-gFUot'

if not os.path.isdir('iscors-net'):
    !git clone --depth 1 -b {BRANCH} {REPO}
else:
    !git -C iscors-net pull

os.chdir('iscors-net')
print('Working dir:', os.getcwd())

!pip install -q -r requirements.txt scipy
print('Dependencies installed.')

In [ ]:
import os

# Always regenerate — ensures new background design (near-static) is used
video_path = './data/test_synthetic_cell.tif'
if os.path.exists(video_path):
    os.remove(video_path)
    print('Removed old video.')

os.makedirs('./data', exist_ok=True)
!python utils/generate_test_video.py

In [ ]:
!python train_phys_recon.py

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

result_files = sorted(glob.glob('./result/*.png'))
print(f'Result images ({len(result_files)}):', [os.path.basename(f) for f in result_files])

for path in result_files:
    img = mpimg.imread(path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.tight_layout()
    plt.show()

In [ ]:
import zipfile, os, glob
from google.colab import files

zip_name = f'results_{VERSION}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pattern in [f'./result/*{VERSION}*', f'./checkpoint/*{VERSION}*']:
        for path in glob.glob(pattern):
            zf.write(path, os.path.relpath(path, '.'))
    for path in glob.glob('./result/*.png'):
        arcname = os.path.relpath(path, '.')
        if arcname not in zf.namelist():
            zf.write(path, arcname)

print(f'Created {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)

---

## Real Cell Analysis

**Workflow:**
1. **real-setup** — Mount Google Drive, unzip `large_file.zip`, locate files.
2. **real-mat** — Decode `Output_iSCORS_map.mat` → TIF reference maps (not fed to model).
3. **real-preprocess** — Normalise video: ÷ temporal median → ÷ per-frame Gaussian (σ=4).
4. **real-inference** — Run trained model on preprocessed video.
5. **real-compare** — Side-by-side: model predictions vs traditional iSCORS reference.


In [ ]:
# ── Real Cell Analysis: Mount Drive & Extract ─────────────────────────────
from google.colab import drive
import zipfile, os

drive.mount('/content/drive', force_remount=False)

ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'
EXTRACT_DIR = '/content/real_data'
SAVE_DIR    = '/content/drive/MyDrive/iscors_test'

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,    exist_ok=True)

print(f'Extracting {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extraction done.')

def find_file(root, name):
    for dirpath, _, files in os.walk(root):
        if name in files:
            return os.path.join(dirpath, name)
    return None

VIDEO_PATH = find_file(EXTRACT_DIR, 'COBRI_rarw_video.tif')
MAT_PATH   = find_file(EXTRACT_DIR, 'Output_iSCORS_map.mat')

print(f'Video : {VIDEO_PATH}')
print(f'MAT   : {MAT_PATH}')
assert VIDEO_PATH, 'COBRI_rarw_video.tif not found in zip!'
assert MAT_PATH,   'Output_iSCORS_map.mat not found in zip!'


In [ ]:
# ── Decode Output_iSCORS_map.mat → TIF (reference only, not fed to model) ──
import numpy as np
import tifffile

# Try scipy.io (MATLAB v5/v7); fall back to h5py (v7.3 / HDF5)
gamma_ref = alpha_ref = None
try:
    import scipy.io as sio
    mat = sio.loadmat(MAT_PATH)
    data = {k: v for k, v in mat.items() if not k.startswith('_')}
    print('Loaded via scipy.io.  Fields:', list(data.keys()))
    _h5 = False
except Exception as e:
    print(f'scipy.io failed ({e}), trying h5py ...')
    import h5py
    _h5_file = h5py.File(MAT_PATH, 'r')
    data = {k: _h5_file[k] for k in _h5_file}
    print('Loaded via h5py.  Fields:', list(data.keys()))
    _h5 = True

def _find(d, *names):
    lmap = {k.lower(): k for k in d}
    for n in names:
        if n.lower() in lmap:
            return np.array(d[lmap[n.lower()]]).squeeze().astype(np.float32)
    return None

gamma_ref = _find(data, 'gamma', 'Gamma', 'gamma_map', 'D')
alpha_ref = _find(data, 'alpha', 'Alpha', 'alpha_map', 'beta', 'anomalous_exp')

if _h5:
    _h5_file.close()

for name, arr in [('gamma_ref', gamma_ref), ('alpha_ref', alpha_ref)]:
    if arr is not None:
        print(f'{name}: shape={arr.shape}  range=[{arr.min():.4f}, {arr.max():.4f}]')
    else:
        print(f'{name}: field not auto-detected — inspect data keys above and set manually')

# Save reference TIFs to Drive
if gamma_ref is not None:
    p = os.path.join(SAVE_DIR, 'iscors_gamma_ref.tif')
    tifffile.imwrite(p, gamma_ref)
    print(f'Saved -> {p}')
if alpha_ref is not None:
    p = os.path.join(SAVE_DIR, 'iscors_alpha_ref.tif')
    tifffile.imwrite(p, alpha_ref)
    print(f'Saved -> {p}')


In [ ]:
# ── Preprocess Real Video ──────────────────────────────────────────────────
# Step 1 — flat-field:  divide by per-pixel temporal median
# Step 2 — background:  divide each frame by Gaussian-smoothed self (sigma=4)
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

print(f'Loading first 2000 frames from {VIDEO_PATH} ...')
video_raw = tifffile.imread(VIDEO_PATH, key=range(2000)).astype(np.float32)
T, H, W = video_raw.shape
print(f'Video shape: T={T}  H={H}  W={W}  (dtype float32)')
print(f'Intensity range: [{video_raw.min():.1f}, {video_raw.max():.1f}]  '
      f'mean={video_raw.mean():.1f}')

# ── Plot raw frame 0 ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(video_raw[0], cmap='gray')
plt.colorbar(im, ax=ax, label='Intensity')
ax.set_title('Frame 0 — raw')
ax.axis('off')
plt.tight_layout()
plt.show()

# ── Step 1: flat-field ──────────────────────────────────────────────────────
print('\nStep 1: flat-field (/ temporal median) ...')
median_xy = np.median(video_raw, axis=0)                    # (H, W)
video_ff  = video_raw / (median_xy[np.newaxis] + 1e-10)
print(f'  After flat-field: mean={video_ff.mean():.4f}  std={video_ff.std():.6f}')

# ── Step 2: per-frame Gaussian background division ──────────────────────────
print('Step 2: Gaussian background division (sigma=4, per frame) ...')
video_proc = np.empty_like(video_ff)
for t in range(T):
    bg = gaussian_filter(video_ff[t], sigma=4)
    video_proc[t] = video_ff[t] / (bg + 1e-10)
    if t % max(1, T // 5) == 0:
        print(f'  frame {t:4d}/{T}')

print(f'\nProcessed: mean={video_proc.mean():.4f}  std={video_proc.std():.6f}  '
      f'range=[{video_proc.min():.4f}, {video_proc.max():.4f}]')

# ── Plot processed frame 0 ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, frame, title in zip(axes,
    [video_ff[0], video_proc[0]],
    ['Frame 0 — after flat-field', 'Frame 0 — after flat-field + Gaussian BG removal']):
    p1, p99 = np.percentile(frame, 1), np.percentile(frame, 99)
    im = ax.imshow(frame, cmap='gray', vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Preprocessing done.')


In [ ]:
# ── Real Video Inference ───────────────────────────────────────────────────
import sys, os, numpy as np

# Clear HuggingFace 'datasets' from cache so local package is found
for _k in list(sys.modules.keys()):
    if _k == 'datasets' or _k.startswith('datasets.'):
        del sys.modules[_k]

import torch
import matplotlib.pyplot as plt
from datasets.phys_recon_dataset import PhysReconDataset
from models.pissl_tau_encoder    import PISSLTauEncoder

RECON_TAUS = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
CKPT_PATH  = f'./checkpoint/pissl_phys_recon_{VERSION}.pth'
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Compute G_empirical (mode=eval → full frame, no blind-spot)
print('Computing G_empirical ...')
real_ds = PhysReconDataset(
    video_tensor = video_proc,
    recon_taus   = RECON_TAUS,
    patch_size   = 64,
    mode         = 'eval',
)

# Load trained model
print(f'Loading: {CKPT_PATH}')
model = PISSLTauEncoder(recon_taus=RECON_TAUS, predict_amplitude=False).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

# Pad to multiple of 8 (3x MaxPool2d)
full_input = real_ds[0]                                    # (K, H, W)
K, Hv, Wv  = full_input.shape
pad_h = (8 - Hv % 8) % 8
pad_w = (8 - Wv % 8) % 8
if pad_h or pad_w:
    import torch.nn.functional as F
    full_input = F.pad(full_input, (0, pad_w, 0, pad_h))
    print(f'Padded: {Hv}x{Wv} -> {Hv+pad_h}x{Wv+pad_w}')

with torch.no_grad():
    preds_real = model(full_input.unsqueeze(0).to(device))  # (1, 2, H', W')

gamma_pred = preds_real[0, 0].cpu().numpy()[:Hv, :Wv]
alpha_pred = preds_real[0, 1].cpu().numpy()[:Hv, :Wv]
cell_mask  = real_ds.cell_mask

# ── Plot inference maps ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, cmap, vmin, vmax, title in [
    (axes[0], np.where(cell_mask, gamma_pred, np.nan), 'magma',   0,   1.0, f'Model gamma [{VERSION}]'),
    (axes[1], np.where(cell_mask, alpha_pred, np.nan), 'viridis', 0.5, 2.0, f'Model alpha [{VERSION}]'),
]:
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Model Inference — Real Cell', fontsize=13)
plt.tight_layout()
plt.show()

# Statistics
g_cell = gamma_pred[cell_mask]; a_cell = alpha_pred[cell_mask]
print(f'\nCell pixels: {cell_mask.sum()}')
print(f'Model gamma: mean={g_cell.mean():.3f}  std={g_cell.std():.3f}  '
      f'range=[{g_cell.min():.3f}, {g_cell.max():.3f}]')
print(f'Model alpha: mean={a_cell.mean():.3f}  std={a_cell.std():.3f}  '
      f'range=[{a_cell.min():.3f}, {a_cell.max():.3f}]')
print('Inference done.')


In [ ]:
# ── Compare Model vs Traditional iSCORS + Save Results ─────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tifffile, os
from skimage.transform import resize as sk_resize

cell_mask = real_ds.cell_mask

def _match(arr, shape):
    return arr if arr.shape == shape else sk_resize(
        arr, shape, preserve_range=True, anti_aliasing=True).astype(np.float32)

# ── Side-by-side comparison ──────────────────────────────────────────────────
panels = [
    (f'Model gamma [{VERSION}]',  np.where(cell_mask, gamma_pred, np.nan), 'magma',   0,   1.0),
    (f'Model alpha [{VERSION}]',  np.where(cell_mask, alpha_pred, np.nan), 'viridis', 0.5, 2.0),
]
if gamma_ref is not None:
    gr = _match(gamma_ref, gamma_pred.shape)
    panels.append(('iSCORS gamma (ref)', gr, 'magma',
                   np.nanpercentile(gr, 1), np.nanpercentile(gr, 99)))
if alpha_ref is not None:
    ar = _match(alpha_ref, alpha_pred.shape)
    panels.append(('iSCORS alpha (ref)', ar, 'viridis',
                   np.nanpercentile(ar, 1), np.nanpercentile(ar, 99)))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
if len(panels) == 1: axes = [axes]
for ax, (title, data, cmap, vmin, vmax) in zip(axes, panels):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.suptitle('Model vs Traditional iSCORS', fontsize=13)
plt.tight_layout()
plt.show()

# ── Scatter: model vs iSCORS (on cell pixels) ────────────────────────────────
if gamma_ref is not None and alpha_ref is not None:
    gr_m = _match(gamma_ref, gamma_pred.shape)[cell_mask]
    ar_m = _match(alpha_ref,  alpha_pred.shape)[cell_mask]
    g_model = gamma_pred[cell_mask]
    a_model = alpha_pred[cell_mask]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, x, y, param in [(axes[0], gr_m, g_model, 'gamma'),
                             (axes[1], ar_m, a_model, 'alpha')]:
        ax.scatter(x, y, s=0.5, alpha=0.3)
        lim = [min(x.min(), y.min()), max(x.max(), y.max())]
        ax.plot(lim, lim, 'r--', lw=1, label='y=x')
        ax.set_xlabel(f'iSCORS {param}'); ax.set_ylabel(f'Model {param}')
        ax.set_title(f'{param}: model vs iSCORS')
        ax.legend()
    plt.tight_layout()
    plt.show()

# ── Numerical comparison ─────────────────────────────────────────────────────
stats_lines = [f'=== Real Cell Analysis [{VERSION}] ===',
               f'Video: {VIDEO_PATH}',
               f'Frames used: {T}  |  Cell pixels: {cell_mask.sum()}',
               '',
               '--- Model predictions ---',
               f'  gamma: mean={g_cell.mean():.4f}  std={g_cell.std():.4f}  '
               f'range=[{g_cell.min():.4f}, {g_cell.max():.4f}]',
               f'  alpha: mean={a_cell.mean():.4f}  std={a_cell.std():.4f}  '
               f'range=[{a_cell.min():.4f}, {a_cell.max():.4f}]']

if gamma_ref is not None and alpha_ref is not None:
    from scipy.stats import pearsonr
    gr_m = _match(gamma_ref, gamma_pred.shape)[cell_mask]
    ar_m = _match(alpha_ref,  alpha_pred.shape)[cell_mask]
    mae_g = np.abs(gamma_pred[cell_mask] - gr_m).mean()
    mae_a = np.abs(alpha_pred[cell_mask] - ar_m).mean()
    r_g   = pearsonr(gamma_pred[cell_mask], gr_m)[0]
    r_a   = pearsonr(alpha_pred[cell_mask], ar_m)[0]
    stats_lines += ['',
        '--- vs Traditional iSCORS ---',
        f'  gamma MAE={mae_g:.4f}   Pearson r={r_g:.4f}',
        f'  alpha MAE={mae_a:.4f}   Pearson r={r_a:.4f}']

print('\n'.join(stats_lines))

# ── Save everything to Drive ─────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)

# TIFs
_gamma_out = np.where(cell_mask, gamma_pred, 0).astype(np.float32)
_alpha_out = np.where(cell_mask, alpha_pred, 0).astype(np.float32)
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_gamma_{VERSION}.tif'), _gamma_out)
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_alpha_{VERSION}.tif'), _alpha_out)

# Stats text
_stats_path = os.path.join(SAVE_DIR, f'real_stats_{VERSION}.txt')
with open(_stats_path, 'w') as f:
    f.write('\n'.join(stats_lines) + '\n')

# Save comparison figure
_cmp_fig_path = os.path.join(SAVE_DIR, f'real_comparison_{VERSION}.png')
fig.savefig(_cmp_fig_path, dpi=120, bbox_inches='tight')
print(f'\nSaved TIFs, stats, and comparison figure -> {SAVE_DIR}/')


In [ ]:
# ── Download Real Inference Results as ZIP ──────────────────────────────────
import zipfile, os, glob
from google.colab import files

zip_name = f'inference_real_{VERSION}.zip'

# Collect: all PNGs from result/ that belong to this session,
# plus TIFs and stats saved to Drive SAVE_DIR
collected = []

# Local result/ PNGs (synthetic training outputs already in first zip)
# Real-analysis outputs saved to Drive
for pattern in [
    os.path.join(SAVE_DIR, f'*{VERSION}*'),
    os.path.join(SAVE_DIR, 'iscors_gamma_ref.tif'),
    os.path.join(SAVE_DIR, 'iscors_alpha_ref.tif'),
]:
    collected.extend(glob.glob(pattern))

collected = sorted(set(collected))
print(f'Files to zip ({len(collected)}):')
for p in collected:
    print(f'  {os.path.basename(p)}  ({os.path.getsize(p)/1024:.1f} KB)')

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in collected:
        zf.write(p, os.path.basename(p))

print(f'\nCreated {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)
